In [9]:
import pandas as pd
import os

# ============================================================
# SETTINGS
# ============================================================

# Path to warehouse stock file (user must update this)
WAREHOUSE_PATH = r"C:\Users\amira\OneDrive\Desktop\ارصدة.xlsx"

# Path to branches sales/stock file (user must update this)
BRANCHES_PATH  = r"C:\Users\amira\OneDrive\Desktop\مبيعات.xlsx"

# ============================
# 📅 REPORT SETTINGS
# ============================

# Report start and end dates (user-defined)
START_DATE     = "2026-03-01"
END_DATE       = "2026-06-01"

# Coverage days used to calculate minimum stock levels
COVERAGE_DAYS  = 90
# عدد أيام الأمان فوق التغطية الأساسية (Buffer)
SAFETY_DAYS    = 14

# حدود تصنيف الفروع حسب المبيعات التراكمية (ABC Analysis)
ABC_A_THRESH   = 40   # أعلى 40% من الفروع = فروع A
ABC_B_THRESH   = 75   # حتى 75% = فروع B
                     # الباقي = فروع C

# أوزان أهمية مخزون الفروع (A=100%, B=50%, C=0%)
WEIGHT_A       = 1.00
WEIGHT_B       = 0.50
WEIGHT_C       = 0.00

# معامل زيادة الطلب للأصناف السريعة (Boost للـ FAST items)
FAST_MULTIPLIER = 1.30

# ============================================================
# LOAD & CLEAN
# ============================================================

report_days = (pd.to_datetime(END_DATE) - pd.to_datetime(START_DATE)).days
print(f"\n✔️ عدد أيام التقرير: {report_days} يوم\n")

warehouse = pd.read_excel(WAREHOUSE_PATH)
branches  = pd.read_excel(BRANCHES_PATH)
print("\n🔥 Columns in branches file:")

warehouse = warehouse.drop_duplicates(subset=["ITEM_CODE"])
branches  = branches.drop_duplicates(subset=["ITEM_CODE", "STORE_NAME"])

warehouse = warehouse.rename(columns={"NAME_E": "ITEM_NAME", "BAL": "WAREHOUSE_BAL"})

branches  = branches.dropna(subset=["ITEM_CODE"])
warehouse = warehouse.dropna(subset=["ITEM_CODE"])

branches["ITEM_CODE"]  = branches["ITEM_CODE"].astype(int)
warehouse["ITEM_CODE"] = warehouse["ITEM_CODE"].astype(int)

merged = branches.merge(
    warehouse[["ITEM_CODE", "WAREHOUSE_BAL"]],
    on="ITEM_CODE",
    how="left"
)
merged = merged.drop_duplicates(subset=["ITEM_CODE", "STORE_NAME"])

print("✔️ تم تحميل الملفات والدمج")

# ============================================================
# ABC BRANCH CLASSIFICATION
# ============================================================

branch_sales = merged.groupby("STORE_NAME")["SALES_QTY"].sum().reset_index()
branch_sales = branch_sales.sort_values("SALES_QTY", ascending=False)
branch_sales["CUM_PERCENT"] = (
    branch_sales["SALES_QTY"].cumsum() / branch_sales["SALES_QTY"].sum() * 100
)

def classify_branch(p):
    if p <= ABC_A_THRESH: return "A"
    elif p <= ABC_B_THRESH: return "B"
    else: return "C"

branch_sales["CLASS"] = branch_sales["CUM_PERCENT"].apply(classify_branch)
merged = merged.merge(branch_sales[["STORE_NAME", "CLASS"]], on="STORE_NAME", how="left")

a = (branch_sales["CLASS"] == "A").sum()
b = (branch_sales["CLASS"] == "B").sum()
c = (branch_sales["CLASS"] == "C").sum()
print(f"✔️ تصنيف الفروع: A={a} | B={b} | C={c}")

# ============================================================
# ITEM SUMMARY
# ============================================================

item_summary = merged.groupby("ITEM_CODE").agg(
    ITEM_NAME     = ("ITEM_NAME",     "first"),
    SALES_QTY     = ("SALES_QTY",     "sum"),
    BRANCH_BAL    = ("BALANCE",       "sum"),
    WAREHOUSE_BAL = ("WAREHOUSE_BAL", "first"),
    UNIT_COST     = ("UNIT_COST",     "first"),
    SALES_PRICE   = ("SALES_PRICE",   "first"),
).reset_index()

item_summary["WAREHOUSE_BAL"] = item_summary["WAREHOUSE_BAL"].fillna(0)

# ============================================================
# DAILY CONSUMPTION & NEED (كل الفروع)
# ============================================================

item_summary["DAILY_CONSUMPTION"] = item_summary["SALES_QTY"] / report_days
item_summary["PERIOD_NEED"]       = item_summary["DAILY_CONSUMPTION"] * (COVERAGE_DAYS + SAFETY_DAYS)

# ============================================================
# USEFUL STOCK (A=100%, B=50%, C=0%)
# ============================================================

merged["BRANCH_USEFUL"] = merged.apply(
    lambda r: r["BALANCE"] * (
        WEIGHT_A if r["CLASS"] == "A" else
        WEIGHT_B if r["CLASS"] == "B" else
        WEIGHT_C
    ), axis=1
)

branch_useful = merged.groupby("ITEM_CODE")["BRANCH_USEFUL"].sum().reset_index()
branch_useful = branch_useful.rename(columns={"BRANCH_USEFUL": "USEFUL_BRANCH_STOCK"})
item_summary  = item_summary.merge(branch_useful, on="ITEM_CODE", how="left")

item_summary["TOTAL_USEFUL_STOCK"] = (
    item_summary["USEFUL_BRANCH_STOCK"].fillna(0) + item_summary["WAREHOUSE_BAL"]
)

# ============================================================
# WEAK BRANCH LEAKAGE
# ============================================================

weak_stock = (
    merged[merged["CLASS"] == "C"]
    .groupby("ITEM_CODE")["BALANCE"].sum()
    .reset_index()
    .rename(columns={"BALANCE": "WEAK_BRANCH_STOCK"})
)
item_summary = item_summary.merge(weak_stock, on="ITEM_CODE", how="left")
item_summary["WEAK_BRANCH_STOCK"] = item_summary["WEAK_BRANCH_STOCK"].fillna(0)

# ============================================================
# TURNOVER
# ============================================================

item_summary["TURNOVER_PROXY"] = (
    item_summary["SALES_QTY"] / (item_summary["TOTAL_USEFUL_STOCK"] + 1)
)

def classify_turnover(x):
    if x >= 6:   return "FAST"
    elif x >= 3: return "MEDIUM"
    elif x >= 1: return "SLOW"
    else:        return "DEAD"

item_summary["TURNOVER_CLASS"] = item_summary["TURNOVER_PROXY"].apply(classify_turnover)

# ============================================================
# PURCHASE QTY
# ============================================================

item_summary["PURCHASE_QTY"] = (
    item_summary["PERIOD_NEED"] - item_summary["TOTAL_USEFUL_STOCK"]
).apply(lambda x: max(round(x), 0))

def tuned_purchase(row):
    qty = row["PURCHASE_QTY"]
    if qty <= 0: return 0
    if row["TURNOVER_CLASS"] == "FAST": return round(qty * FAST_MULTIPLIER)
    return qty

item_summary["PURCHASE_QTY_TUNED"] = item_summary.apply(tuned_purchase, axis=1)

# ============================================================
# FLAGS
# ============================================================

item_summary["SLOW_MOVING"] = item_summary.apply(
    lambda r: "YES" if r["DAILY_CONSUMPTION"] == 0 and r["TOTAL_USEFUL_STOCK"] > 0 else "NO",
    axis=1
)

def purchase_priority(row):
    if row["PURCHASE_QTY"] >= 50 or row["PERIOD_NEED"] >= 150: return "RED"
    elif row["PURCHASE_QTY"] >= 20: return "ORANGE"
    elif row["PURCHASE_QTY"] > 0:   return "YELLOW"
    else:                            return "GREEN"

item_summary["PRIORITY_COLOR"] = item_summary.apply(purchase_priority, axis=1)

# ============================================================
# OUTPUT SHEETS
# ============================================================

slow_items = item_summary[item_summary["SLOW_MOVING"] == "YES"]["ITEM_CODE"].tolist()

slow_branches = (
    merged[(merged["ITEM_CODE"].isin(slow_items)) & (merged["BALANCE"] > 0)]
    [["ITEM_CODE", "ITEM_NAME", "STORE_NAME", "BALANCE"]]
    .rename(columns={"BALANCE": "BRANCH_BAL"})
)

leakage_branches = (
    merged[(merged["CLASS"] == "C") & (merged["BALANCE"] > 0)]
    [["ITEM_CODE", "ITEM_NAME", "STORE_NAME", "BALANCE", "CLASS"]]
    .rename(columns={"BALANCE": "BRANCH_BAL"})
)

need_purchase = item_summary[item_summary["PURCHASE_QTY"] > 0][[
    "ITEM_CODE", "ITEM_NAME", "PURCHASE_QTY", "PURCHASE_QTY_TUNED",
    "PERIOD_NEED", "TOTAL_USEFUL_STOCK", "PRIORITY_COLOR"
]]

purchases_df = item_summary[[
    "ITEM_CODE", "ITEM_NAME", "SALES_QTY", "DAILY_CONSUMPTION", "PERIOD_NEED",
    "TOTAL_USEFUL_STOCK", "WEAK_BRANCH_STOCK", "PURCHASE_QTY", "PURCHASE_QTY_TUNED",
    "TURNOVER_CLASS", "PRIORITY_COLOR"
]].copy()

warehouse_df_out = item_summary[[
    "ITEM_CODE", "ITEM_NAME", "WAREHOUSE_BAL", "USEFUL_BRANCH_STOCK",
    "TOTAL_USEFUL_STOCK", "SLOW_MOVING", "WEAK_BRANCH_STOCK"
]].copy()

branch_class_df = branch_sales[["STORE_NAME", "SALES_QTY", "CUM_PERCENT", "CLASS"]].copy()
branch_class_df["CUM_PERCENT"] = branch_class_df["CUM_PERCENT"].round(1)
branch_class_df.columns = ["Branch", "Total Sales", "Cumulative %", "Class"]

slow_list = item_summary[item_summary["SLOW_MOVING"] == "YES"][[
    "ITEM_CODE", "ITEM_NAME", "TOTAL_USEFUL_STOCK"
]]

summary_df = pd.DataFrame({
    "Metric": [
        "Total Items",
        "Items with Purchase Need",
        "Slow Moving Items",
        "Items with Leakage in Weak Branches",
        "ABC Thresholds (A / B / C)",
        "Coverage Days",
        "Safety Stock Buffer (days)",
    ],
    "Value": [
        item_summary["ITEM_CODE"].nunique(),
        (item_summary["PURCHASE_QTY"] > 0).sum(),
        (item_summary["SLOW_MOVING"] == "YES").sum(),
        (item_summary["WEAK_BRANCH_STOCK"] > 0).sum(),
        f"{ABC_A_THRESH}% / {ABC_B_THRESH}% / rest",
        COVERAGE_DAYS,
        SAFETY_DAYS,
    ]
})

# ============================================================
# EXCEL OUTPUT
# ============================================================

user_home = os.path.expanduser("~")
for path in [
    os.path.join(user_home, "Desktop"),
    os.path.join(user_home, "سطح المكتب"),
    os.path.join(user_home, "OneDrive", "Desktop"),
    os.path.join(user_home, "OneDrive", "سطح المكتب"),
]:
    if os.path.exists(path):
        desktop = path
        break

output_path = os.path.join(desktop, "تقرير مشتريات و مخزن.xlsx")

sheets = {
    "Purchases":            purchases_df,
    "NeedPurchase":         need_purchase,
    "Warehouse":            warehouse_df_out,
    "SlowBranches":         slow_branches,
    "SlowItems":            slow_list,
    "LeakageBranches":      leakage_branches,
    "BranchClassification": branch_class_df,
    "Summary":              summary_df,
}

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    workbook = writer.book
    red_fmt    = workbook.add_format({'bg_color': '#FF6B6B'})
    orange_fmt = workbook.add_format({'bg_color': '#FFD580'})
    yellow_fmt = workbook.add_format({'bg_color': '#FFF59D'})
    green_fmt  = workbook.add_format({'bg_color': '#C8E6C9'})

    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        rows, cols = df.shape
        if rows > 0:
            ws.add_table(0, 0, rows, cols - 1, {
                'columns': [{'header': c} for c in df.columns],
                'style': 'Table Style Medium 9'
            })

    # Purchases
    ws   = writer.sheets["Purchases"]
    rows = len(purchases_df)
    for val, fmt in [("FAST", green_fmt), ("MEDIUM", yellow_fmt), ("SLOW", orange_fmt), ("DEAD", red_fmt)]:
        ws.conditional_format(f'J2:J{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': val, 'format': fmt})
    for val, fmt in [("RED", red_fmt), ("ORANGE", orange_fmt), ("YELLOW", yellow_fmt), ("GREEN", green_fmt)]:
        ws.conditional_format(f'K2:K{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': val, 'format': fmt})

    # NeedPurchase
    ws   = writer.sheets["NeedPurchase"]
    rows = len(need_purchase)
    for val, fmt in [("RED", red_fmt), ("ORANGE", orange_fmt), ("YELLOW", yellow_fmt), ("GREEN", green_fmt)]:
        ws.conditional_format(f'G2:G{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': val, 'format': fmt})

    # Warehouse
    ws   = writer.sheets["Warehouse"]
    rows = len(warehouse_df_out)
    ws.conditional_format(f'F2:F{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': 'YES', 'format': red_fmt})
    ws.conditional_format(f'F2:F{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': 'NO',  'format': green_fmt})

    # BranchClassification
    ws   = writer.sheets["BranchClassification"]
    rows = len(branch_class_df)
    for val, fmt in [("A", green_fmt), ("B", yellow_fmt), ("C", red_fmt)]:
        ws.conditional_format(f'D2:D{rows+1}', {'type': 'text', 'criteria': 'containing', 'value': val, 'format': fmt})

print("==============================")
print("✔️ تم إنشاء ملف Excel النهائي")
print(output_path)
print("==============================\n")



✔️ عدد أيام التقرير: 92 يوم


🔥 Columns in branches file:
✔️ تم تحميل الملفات والدمج
✔️ تصنيف الفروع: A=2 | B=6 | C=17
✔️ تم إنشاء ملف Excel النهائي
C:\Users\amira\OneDrive\Desktop\تقرير مشتريات و مخزن.xlsx

